# TRELLIS.2 - Bild zu 3D für OpenTTS

**Microsoft's Image-to-3D mit vollen PBR-Texturen (MIT Lizenz)**

**Workflow:**
1. Setup ausführen (einmal, ~5 min)
2. Bild hochladen
3. 3D-Modell generieren
4. Herunterladen

**Wichtig:** Benötigt A100 GPU (Colab Pro) - mindestens 24GB VRAM!

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DutchMaxwell/openTTS/blob/main/tools/3d-generation/trellis2_colab.ipynb)

---
## Schritt 1: Setup (einmal ausführen)
Dauert ca. 5 Minuten. Danach kannst du beliebig viele Bilder konvertieren.

In [ ]:
#@title 1. Setup ausführen {display-mode: "form"}

#@markdown ### Optionen:
debug_mode = True #@param {type:"boolean"}

import os
import time
import sys
import subprocess

def run_cmd(cmd, desc, show_output=False, critical=False):
    """Führt Kommando aus mit Timer und optionalem Output"""
    print(f"\n⏳ {desc}...")
    start = time.time()
    
    process = subprocess.Popen(
        cmd, shell=True, 
        stdout=subprocess.PIPE, 
        stderr=subprocess.STDOUT,
        text=True
    )
    output_lines = []
    for line in process.stdout:
        output_lines.append(line)
        if show_output:
            print(f"   {line}", end="")
    process.wait()
    success = process.returncode == 0
    
    elapsed = time.time() - start
    mins, secs = int(elapsed // 60), int(elapsed % 60)
    status = "✅" if success else "❌"
    print(f"{status} {desc} [{mins:02d}:{secs:02d}]")
    
    if not success and critical:
        print("\n⚠️  KRITISCHER FEHLER - Output:")
        for line in output_lines[-30:]:  # Letzte 30 Zeilen
            print(f"   {line}", end="")
        raise RuntimeError(f"Installation von {desc} fehlgeschlagen!")
    
    return success

# GPU Check
print("=" * 50)
print("SYSTEM CHECK")
print("=" * 50)

result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], 
                      capture_output=True, text=True)
gpu_info = result.stdout.strip()
print(f"GPU: {gpu_info}")

if "A100" in gpu_info or "H100" in gpu_info or "L40" in gpu_info:
    print("✅ GPU OK (24GB+ VRAM)")
else:
    print("⚠️  TRELLIS.2 braucht ~24GB VRAM. A100 empfohlen!")

# Check Colab's default PyTorch
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")
print(f"Python: {sys.version.split()[0]}")

# Installation
if not os.path.exists('/content/TRELLIS.2/trellis2'):
    total_start = time.time()
    
    print("\n" + "=" * 50)
    print("INSTALLATION")
    print("=" * 50)
    
    # Build tools
    run_cmd("pip install ninja setuptools wheel", "Build-Tools", show_output=debug_mode)
    
    # System deps
    run_cmd("apt-get update && apt-get install -y libegl1-mesa-dev libgl1-mesa-dev", 
            "System-Dependencies", show_output=debug_mode)
    
    # Clone TRELLIS.2
    run_cmd("git clone --depth 1 https://github.com/microsoft/TRELLIS.2.git /content/TRELLIS.2",
            "TRELLIS.2 Repository", show_output=debug_mode)
    os.chdir("/content/TRELLIS.2")
    
    # CUDA Extensions
    print("\n" + "-" * 40)
    print("CUDA Extensions")
    print("-" * 40)
    
    os.makedirs("/tmp/extensions", exist_ok=True)
    
    # nvdiffrast
    run_cmd("pip install git+https://github.com/NVlabs/nvdiffrast.git --no-build-isolation",
            "nvdiffrast", show_output=debug_mode, critical=True)
    
    # CuMesh
    run_cmd("git clone https://github.com/JeffreyXiang/CuMesh.git /tmp/extensions/CuMesh --recursive",
            "CuMesh klonen", show_output=debug_mode)
    run_cmd("pip install /tmp/extensions/CuMesh --no-build-isolation",
            "CuMesh", show_output=debug_mode, critical=True)
    
    # Triton + FlexGEMM
    run_cmd("pip install triton", "Triton", show_output=debug_mode)
    run_cmd("git clone https://github.com/JeffreyXiang/FlexGEMM.git /tmp/extensions/FlexGEMM --recursive",
            "FlexGEMM klonen", show_output=debug_mode)
    run_cmd("pip install /tmp/extensions/FlexGEMM --no-build-isolation",
            "FlexGEMM", show_output=debug_mode, critical=True)
    
    # O-Voxel (WICHTIG: Immer mit Output zeigen!)
    print("\n⏳ O-Voxel installieren (mit vollem Output)...")
    result = subprocess.run(
        "pip install /content/TRELLIS.2/o-voxel --no-build-isolation -v",
        shell=True, capture_output=True, text=True
    )
    if result.returncode != 0:
        print("❌ O-Voxel Installation fehlgeschlagen!")
        print("\nSTDOUT:")
        print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
        print("\nSTDERR:")
        print(result.stderr[-3000:] if len(result.stderr) > 3000 else result.stderr)
        raise RuntimeError("O-Voxel konnte nicht installiert werden!")
    else:
        print("✅ O-Voxel installiert")
    
    # Python packages + xformers
    run_cmd("pip install xformers", "xformers", show_output=debug_mode)
    run_cmd("pip install imageio imageio-ffmpeg tqdm easydict opencv-python-headless trimesh transformers pillow huggingface_hub einops safetensors accelerate diffusers",
            "Python-Pakete", show_output=debug_mode)
    
    # VERIFIKATION
    print("\n" + "-" * 40)
    print("VERIFIKATION")
    print("-" * 40)
    
    modules = ['nvdiffrast', 'cumesh', 'flex_gemm', 'o_voxel', 'xformers']
    all_ok = True
    for mod in modules:
        try:
            __import__(mod)
            print(f"✅ {mod}")
        except ImportError as e:
            print(f"❌ {mod}: {e}")
            all_ok = False
    
    if not all_ok:
        raise RuntimeError("Nicht alle Module konnten geladen werden!")
    
    os.environ['ATTN_BACKEND'] = 'xformers'
    sys.path.insert(0, '/content/TRELLIS.2')
    
    total_elapsed = time.time() - total_start
    print(f"\n{'=' * 50}")
    print(f"✅ Installation fertig in {total_elapsed/60:.1f} Minuten!")
    print(f"{'=' * 50}")
else:
    os.chdir("/content/TRELLIS.2")
    sys.path.insert(0, '/content/TRELLIS.2')
    os.environ['ATTN_BACKEND'] = 'xformers'
    print("\n✅ TRELLIS.2 bereits installiert")

os.makedirs('/content/input', exist_ok=True)
os.makedirs('/content/output', exist_ok=True)

print("\n✅ SETUP FERTIG - Weiter zu Schritt 2!")

---
## Schritt 2: Bild hochladen

In [ ]:
#@title 2. Bild hochladen {display-mode: "form"}

from google.colab import files
from pathlib import Path
import shutil
from IPython.display import display, Image as IPImage

print("Wähle ein Bild aus (PNG oder JPG)...")
print("")

uploaded = files.upload()

if uploaded:
    for filename in uploaded.keys():
        dest = f'/content/input/{filename}'
        shutil.move(filename, dest)
        
        print(f"\n✅ Hochgeladen: {filename}")
        print("\nVorschau:")
        display(IPImage(dest, width=300))
        
        current_image = dest
        %store current_image
        
    print("\n" + "=" * 50)
    print("✅ BILD BEREIT - Weiter zu Schritt 3!")
    print("=" * 50)
else:
    print("❌ Kein Bild hochgeladen")

---
## Schritt 3: 3D-Modell generieren

In [ ]:
#@title 3. 3D-Modell generieren {display-mode: "form"}

#@markdown ### Einstellungen:
aufloesung = "512" #@param ["512", "1024", "1536"]

from pathlib import Path
from PIL import Image
import time
import sys
sys.path.insert(0, '/content/TRELLIS.2')

# Get uploaded image
%store -r current_image

try:
    img_path = Path(current_image)
except:
    images = list(Path('/content/input').glob('*.png')) + list(Path('/content/input').glob('*.jpg'))
    if images:
        img_path = images[-1]
    else:
        print("❌ Kein Bild gefunden! Führe zuerst Schritt 2 aus.")
        raise SystemExit()

output_path = f'/content/output/{img_path.stem}.glb'

print("=" * 50)
print(f"Eingabe: {img_path.name}")
print(f"Auflösung: {aufloesung}³")
print("=" * 50)

# Load pipeline
print("\n🔄 Lade TRELLIS.2 Modell...")
from trellis2.pipelines import Trellis2ImageTo3DPipeline

pipeline = Trellis2ImageTo3DPipeline.from_pretrained("microsoft/TRELLIS.2-4B")
pipeline.to("cuda")

# Load image
image = Image.open(img_path).convert("RGB")

# Generate
print(f"\n🔄 Generiere 3D-Modell...")
start = time.time()

mesh = pipeline.run(
    image,
    resolution=int(aufloesung)
)[0]

# Export
mesh.export(output_path)

elapsed = time.time() - start

if Path(output_path).exists():
    size_mb = Path(output_path).stat().st_size / 1e6
    print("\n" + "=" * 50)
    print(f"✅ FERTIG in {elapsed:.0f} Sekunden!")
    print(f"📦 Datei: {Path(output_path).name} ({size_mb:.1f} MB)")
    print("=" * 50)
    print("\nWeiter zu Schritt 4 zum Herunterladen!")
    
    current_output = output_path
    %store current_output
else:
    print("\n❌ Fehler bei der Generierung")

---
## Schritt 4: Herunterladen

In [ ]:
#@title 4. GLB-Datei herunterladen {display-mode: "form"}

from google.colab import files
from pathlib import Path

%store -r current_output

try:
    output_path = Path(current_output)
except:
    outputs = list(Path('/content/output').glob('*.glb'))
    if outputs:
        output_path = outputs[-1]
    else:
        print("❌ Kein 3D-Modell gefunden! Führe zuerst Schritt 3 aus.")
        raise SystemExit()

if output_path.exists():
    size_mb = output_path.stat().st_size / 1e6
    print(f"📦 Download: {output_path.name} ({size_mb:.1f} MB)")
    print("")
    files.download(str(output_path))
    print("\n" + "=" * 50)
    print("✅ Download gestartet!")
    print("")
    print("Nächste Schritte:")
    print("1. GLB-Datei in OpenTTS importieren")
    print("2. Spawn > Import GLB")
    print("3. Positionieren und skalieren")
    print("4. L drücken zum Fixieren")
    print("=" * 50)
else:
    print("❌ Datei nicht gefunden")

---
## Weitere Bilder?

Gehe zurück zu **Schritt 2** und lade das nächste Bild hoch.

Das Modell bleibt geladen - weitere Generierungen sind schneller!

---
## Kein A100? Nutze den HuggingFace Space!

Kostenlos im Browser: **[TRELLIS.2 Demo](https://huggingface.co/spaces/microsoft/TRELLIS.2)**